In [36]:
import numpy as np
import pandas as pd
import pickle
import joblib
import re
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Layer


# ======================================================
# CUSTOM ATTENTION LAYER
# ======================================================

class AttentionLayer(Layer):

    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):

        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], input_shape[-1]),
            initializer='glorot_uniform',
            trainable=True
        )

        self.b = self.add_weight(
            name='attention_bias',
            shape=(input_shape[-1],),
            initializer='zeros',
            trainable=True
        )

        self.u = self.add_weight(
            name='context_vector',
            shape=(input_shape[-1],),
            initializer='glorot_uniform',
            trainable=True
        )

        super(AttentionLayer, self).build(input_shape)

    def call(self, x):

        score = tf.nn.tanh(tf.tensordot(x, self.W, axes=[2,0]) + self.b)

        attention_weights = tf.nn.softmax(
            tf.tensordot(score, self.u, axes=[2,0]),
            axis=1
        )

        context_vector = tf.reduce_sum(
            attention_weights[..., tf.newaxis] * x,
            axis=1
        )

        return context_vector


# ======================================================
# LOAD MODELS
# ======================================================

print("Loading models...")

bilstm_model = load_model(
    "models/disease_prediction_model.h5",
    custom_objects={"AttentionLayer": AttentionLayer}
)

xgb = joblib.load("models/xgboost_model.pkl")

print("Models loaded")


# ======================================================
# LOAD PREPROCESSING
# ======================================================

with open("models/preprocessing.pkl", "rb") as f:
    preprocess = pickle.load(f)

tokenizer = preprocess["tokenizer"]
label_encoder = preprocess["label_encoder"]

max_len = 150


# ======================================================
# LOAD FEATURE LIST
# ======================================================

df = pd.read_csv("dataset/ensemble_dataset.csv")

symptoms = list(df.columns[1:-1])


# ======================================================
# TEXT CLEANING
# ======================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    return text


# ======================================================
# BILSTM PREDICTION
# ======================================================

def bilstm_predict(text):

    text = clean_text(text)

    seq = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(seq, maxlen=max_len)

    probs = bilstm_model.predict(padded, verbose=0)

    return probs


# ======================================================
# TABULAR FEATURE EXTRACTION
# ======================================================

def extract_features(text):

    text = clean_text(text)

    features = []

    for s in symptoms:

        s_clean = s.replace("_", " ")

        if s_clean in text:
            features.append(1)
        else:
            features.append(0)

    return pd.DataFrame([features], columns=symptoms)


# ======================================================
# BEST ENSEMBLE
# ======================================================

if __name__ == "__main__":

    text = "My spirits have been incredibly low, and my pee smells awful. My kidney region hurts a lot, and I can't seem to hold my urine in. I get these unreasonable urges all the time."

    print("\nInput:", text)

    text_probs = bilstm_predict(text)

    features = extract_features(text)

    xgb_probs = xgb.predict_proba(features)


# ------------------------------------------------------
# BiLSTM + XGBoost Ensemble
# ------------------------------------------------------

    final_probs = 0.8 * text_probs + 0.2 * xgb_probs

    pred = np.argmax(final_probs)

    print("\n Prediction")
    print(label_encoder.inverse_transform([pred])[0])

Loading models...
Models loaded

Input: My spirits have been incredibly low, and my pee smells awful. My kidney region hurts a lot, and I can't seem to hold my urine in. I get these unreasonable urges all the time.

 Prediction
urinary tract infection


In [38]:
import numpy as np
import pandas as pd
import pickle
import joblib
import re
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Layer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ======================================================
# CUSTOM ATTENTION LAYER
# ======================================================

class AttentionLayer(Layer):

    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):

        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], input_shape[-1]),
            initializer='glorot_uniform',
            trainable=True
        )

        self.b = self.add_weight(
            name='attention_bias',
            shape=(input_shape[-1],),
            initializer='zeros',
            trainable=True
        )

        self.u = self.add_weight(
            name='context_vector',
            shape=(input_shape[-1],),
            initializer='glorot_uniform',
            trainable=True
        )

        super(AttentionLayer, self).build(input_shape)

    def call(self, x):

        score = tf.nn.tanh(tf.tensordot(x, self.W, axes=[2,0]) + self.b)

        attention_weights = tf.nn.softmax(
            tf.tensordot(score, self.u, axes=[2,0]),
            axis=1
        )

        context_vector = tf.reduce_sum(
            attention_weights[..., tf.newaxis] * x,
            axis=1
        )

        return context_vector


# ======================================================
# LOAD MODELS
# ======================================================

print("Loading models...")

bilstm_model = load_model(
    "models/disease_prediction_model.h5",
    custom_objects={"AttentionLayer": AttentionLayer}
)

xgb = joblib.load("models/xgboost_model.pkl")

print("Models loaded")


# ======================================================
# LOAD PREPROCESSING
# ======================================================

with open("models/preprocessing.pkl", "rb") as f:
    preprocess = pickle.load(f)

tokenizer = preprocess["tokenizer"]
label_encoder = preprocess["label_encoder"]

max_len = 150


# ======================================================
# LOAD FEATURE LIST
# ======================================================

df = pd.read_csv("dataset/ensemble_dataset.csv")

symptoms = list(df.columns[1:-1])


# ======================================================
# LOAD TREATMENT DATASET
# ======================================================

treatment_df = pd.read_csv("dataset/Diseases_SymptomsTreatment.csv")

disease_col = "Name"
treatment_col = "Treatments"

treatment_names = treatment_df[disease_col].str.lower().tolist()


# ======================================================
# CANONICAL DISEASE MAP
# ======================================================

canonical_map = {

"drug reaction":"drug reaction",
"malaria":"malaria",
"allergy":"allergy",
"hypothyroidism":"hypothyroidism",
"psoriasis":"psoriasis",
"gerd":"gerd",
"chronic cholestasis":"cholestasis",
"hepatitis a":"hepatitis a",
"osteoarthritis":"osteoarthritis",
"(vertigo) paroxysmal positional vertigo":"vertigo",
"hypoglycemia":"hypoglycemia",
"acne":"acne",
"diabetes":"diabetes",
"impetigo":"impetigo",
"hypertension":"hypertension",
"peptic ulcer disease":"peptic ulcer",
"dimorphic hemorrhoids(piles)":"hemorrhoids",
"common cold":"normal cold",
"chicken pox":"chickenpox",
"cervical spondylosis":"spondylosis",
"hyperthyroidism":"hyperthyroidism",
"urinary tract infection":"urinary tract infection",
"varicose veins":"varicose veins",
"aids":"aids",
"paralysis (brain hemorrhage)":"brain hemorrhage",
"typhoid":"typhoid",
"hepatitis b":"hepatitis b",
"fungal infection":"fungal infection",
"hepatitis c":"hepatitis c",
"migraine":"migraine",
"bronchial asthma":"asthma",
"alcoholic hepatitis":"alcoholic hepatitis",
"jaundice":"jaundice",
"hepatitis e":"hepatitis e",
"dengue":"dengue",
"hepatitis d":"hepatitis d",
"heart attack":"heart attack",
"pneumonia":"pneumonia",
"arthritis":"arthritis",
"gastroenteritis":"gastroenteritis",
"tuberculosis":"tuberculosis"

}


# ======================================================
# TEXT CLEANING
# ======================================================

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    return text


# ======================================================
# BILSTM PREDICTION
# ======================================================

def bilstm_predict(text):

    text = clean_text(text)

    seq = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(seq, maxlen=max_len)

    probs = bilstm_model.predict(padded, verbose=0)

    return probs


# ======================================================
# TABULAR FEATURE EXTRACTION
# ======================================================

def extract_features(text):

    text = clean_text(text)

    features = []

    for s in symptoms:

        s_clean = s.replace("_", " ")

        if s_clean in text:
            features.append(1)
        else:
            features.append(0)

    return pd.DataFrame([features], columns=symptoms)


# ======================================================
# GET TREATMENT
# ======================================================

def get_treatment(predicted_disease):

    disease = predicted_disease.lower()

    disease = canonical_map.get(disease, disease)

    # exact match
    for _, row in treatment_df.iterrows():

        name = str(row[disease_col]).lower()

        if disease == name:
            return str(row[treatment_col])

    # substring match
    for _, row in treatment_df.iterrows():

        name = str(row[disease_col]).lower()

        if disease in name or name in disease:
            return str(row[treatment_col])

    # semantic similarity
    vectorizer = TfidfVectorizer()

    corpus = [disease] + treatment_names

    tfidf = vectorizer.fit_transform(corpus)

    sim = cosine_similarity(tfidf[0:1], tfidf[1:])[0]

    idx = np.argmax(sim)

    if sim[idx] > 0.65:
        return treatment_df.iloc[idx][treatment_col]

    return "Consult a healthcare professional."


# ======================================================
# TEST BILSTM + XGBOOST ENSEMBLE
# ======================================================

if __name__ == "__main__":

    text = "I have trouble breathing, especially outside. I start to feel hot and start to sweat. I frequently have urinary tract infections and yeast infections."

    print("\nInput:", text)

    text_probs = bilstm_predict(text)

    features = extract_features(text)

    xgb_probs = xgb.predict_proba(features)


# ------------------------------------------------------
# Ensemble
# ------------------------------------------------------

    final_probs = 0.8 * text_probs + 0.2 * xgb_probs

    pred = np.argmax(final_probs)

    disease = label_encoder.inverse_transform([pred])[0]

    print("\nPredicted Disease:", disease)


# ------------------------------------------------------
# Treatment Suggestion
# ------------------------------------------------------

    treatment = get_treatment(disease)

    print("\nSuggested Treatment:")
    print(treatment)

Loading models...
Models loaded

Input: I have trouble breathing, especially outside. I start to feel hot and start to sweat. I frequently have urinary tract infections and yeast infections.

Predicted Disease: diabetes

Suggested Treatment:
Blood sugar monitoring, healthy diet modifications, regular exercise, insulin injections (if needed), close monitoring by healthcare provider
